# Nifty 50 Swing Bot — Weekly Retrain & Deploy

**Run every Sunday.**  
This notebook:
1. Resolves last week's predictions (actual prices vs predicted)
2. Retrains the model with all historical + new data
3. Compares new model vs currently deployed model
4. Deploys if better → saves to Drive + Supabase
5. Generates next week's signals

**Before running:** Add `SUPABASE_URL` and `SUPABASE_KEY` to Colab Secrets (🔑 icon in left sidebar).

**Drive layout** — every run is saved under a date-stamped folder (and the next
run automatically restores from the most recent one):

```
swing_outputs/
└── 2026-06-21/
    ├── models/      ← best_params.json
    ├── registry/    ← trained booster bundle + manifest
    ├── outputs/     ← portfolio + signals (paper-trade state)
    └── data/        ← cached prices
```

In [ ]:
# ── Cell 1: GPU check + environment setup ──────────────────────────────────
import os
import subprocess
from datetime import date

# Confirm GPU
result = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
print(result.stdout or 'No GPU found — will use CPU (slower training)')

# Load Supabase credentials from Colab Secrets.
# New Supabase API key naming (replaces anon/service_role keys): the pipeline
# writes to the DB, so it needs SUPABASE_SECRET_KEY (falls back to the legacy
# SUPABASE_KEY if that's the only secret you've set up).
def _load_secret(*names):
    from google.colab import userdata
    for name in names:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            continue
    return None

try:
    url = _load_secret('SUPABASE_URL')
    key = _load_secret('SUPABASE_SECRET_KEY', 'SUPABASE_KEY')
    if not url or not key:
        raise RuntimeError('SUPABASE_URL / SUPABASE_SECRET_KEY not set in Colab Secrets')
    os.environ['SUPABASE_URL'] = url
    os.environ['SUPABASE_SECRET_KEY'] = key
    print('Supabase credentials loaded from Colab Secrets')
except Exception as e:
    print(f'Could not load Colab Secrets: {e}')
    print('Set SUPABASE_URL and SUPABASE_SECRET_KEY manually below if needed:')
    # os.environ['SUPABASE_URL'] = 'https://xxx.supabase.co'
    # os.environ['SUPABASE_SECRET_KEY'] = 'sb_secret_xxx'

# Configuration
REPO_DIR   = '/content/swing_bot'
DRIVE_BASE = '/content/drive/MyDrive/swing_outputs'   # date-stamped runs live under here
RUN_DATE   = date.today().strftime('%Y-%m-%d')
DRIVE_RUN  = f'{DRIVE_BASE}/{RUN_DATE}'                # this run's folder: swing_outputs/<date>/
HORIZON    = 5       # trading days
TRIALS     = 50      # Optuna trials (reduce to 10 for quick test)
CAPITAL    = 1_000_000   # paper trading capital INR

print(f'Config: horizon={HORIZON}d | trials={TRIALS} | capital=₹{CAPITAL:,.0f}')
print(f'This run saves to: swing_outputs/{RUN_DATE}/')

In [ ]:
# ── Cell 2: Mount Drive + clone/update repo ─────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, re, shutil
from pathlib import Path

# Clone or pull repo
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'   # ← UPDATE THIS

if not Path(REPO_DIR).exists():
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
print(f'Repo ready at {REPO_DIR}')

# Find the most recent previous run to restore state from.
# Prefer the newest date-stamped folder (swing_outputs/<YYYY-MM-DD>/); fall back
# to the legacy flat layout (swing_outputs/models, …) for backward compatibility.
base = Path(DRIVE_BASE)
_date_re = re.compile(r'^\d{4}-\d{2}-\d{2}$')
date_dirs = sorted(
    [p for p in base.glob('*') if p.is_dir() and _date_re.match(p.name)],
    key=lambda p: p.name,
) if base.exists() else []

restore_from = date_dirs[-1] if date_dirs else (base if base.exists() else None)
if restore_from is not None:
    print(f'Restoring previous state from: {restore_from}')
else:
    print('No previous runs on Drive — starting fresh')

# Restore models (best_params) + registry (trained booster) so fast-signals can
# load them, and outputs (portfolio/signals) so paper-trade state carries over.
for folder in ('models', 'registry', 'outputs'):
    dst = Path(REPO_DIR) / folder
    src = (restore_from / folder) if restore_from is not None else None
    if src is not None and src.exists():
        shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
        print(f'  restored {folder}/')
    elif folder == 'models':
        dst.mkdir(parents=True, exist_ok=True)
        print('  models/ — none found, starting fresh')

%cd {REPO_DIR}
import sys; sys.path.insert(0, REPO_DIR)

In [ ]:
# ── Cell 3: Install dependencies ────────────────────────────────────────────
!pip install -q optuna yfinance supabase streamlit plotly python-dotenv
# Note: pandas-ta skipped on Colab due to numpy 2.x compatibility issues
# The pipeline uses fallback TA implementations in that case.

# Verify XGBoost + GPU
import xgboost as xgb
print(f'XGBoost {xgb.__version__}')
try:
    m = xgb.XGBClassifier(device='cuda', n_estimators=1)
    import numpy as np
    m.fit(np.random.randn(10, 2), np.random.randint(0, 2, 10))
    print('GPU (CUDA) training: AVAILABLE')
except Exception as e:
    print(f'GPU not available ({e}) — using CPU')

In [ ]:
# ── Cell 4: Resolve last week's predictions ──────────────────────────────────
import math
import os
from datetime import date
from src.config import Config
from src.data.ingestion import fetch_prices, fetch_index_prices, UNIVERSE
from src.db.supabase_client import get_supabase_client
from src.tracking.outcome_tracker import resolve_outcomes, compute_recent_ic, compute_weekly_ic_series
from src.models.improvement import load_current_model_ic, should_retrain, get_model_version

sb = get_supabase_client()
if sb:
    print('Supabase: connected — runs/predictions will be written to the DB')
else:
    # Pinpoint WHY we fell back, so "JSON-only mode" can't silently surprise you.
    print('Supabase: JSON-only mode — DB writes disabled. Reason:')
    if not (os.getenv('SUPABASE_URL') and
            (os.getenv('SUPABASE_SECRET_KEY') or os.getenv('SUPABASE_KEY'))):
        print('  → credentials not in env. Re-run Cell 1 and check Colab Secrets (🔑) '
              'have notebook access enabled.')
    else:
        try:
            from supabase import create_client
            create_client(os.environ['SUPABASE_URL'],
                          os.environ.get('SUPABASE_SECRET_KEY') or os.environ['SUPABASE_KEY'])
            print('  → connection unexpectedly succeeded here; re-run this cell.')
        except ImportError:
            print('  → supabase package not installed. Re-run Cell 3.')
        except Exception as exc:
            print(f'  → connection failed: {exc}')

# Fetch price data needed to resolve outcomes
print('Fetching price data …')
price_df = fetch_prices(UNIVERSE, start='2024-01-01')  # only recent data needed for resolution
print(f'  {len(price_df):,} rows loaded')

# Resolve
n_resolved = resolve_outcomes(price_df, sb, 'outputs', HORIZON)
print(f'  Resolved {n_resolved} prediction(s)')

# Current deployed model's IC
current_ic, current_run_id = load_current_model_ic(sb)
print(f'  Current deployed model: {current_run_id or "none"}')
print(f'  4-week rolling IC: {current_ic:.4f}' if not math.isnan(current_ic) else '  4-week rolling IC: n/a (insufficient data)')

# IC trend
ic_series = compute_weekly_ic_series(sb, n_weeks=8)
if not ic_series.empty:
    print('\nIC trend (last 8 weeks):')
    print(ic_series.to_string(index=False))

# Emergency check
needs_emergency, msg = should_retrain(current_ic)
if needs_emergency:
    print(f'\n⚠️  EMERGENCY: {msg}')
else:
    print(f'\nModel health: {msg}')

In [ ]:
# ── Cell 5: Full retrain ─────────────────────────────────────────────────────
from src.pipeline.runner import run

new_run_id = get_model_version()
print(f'New run_id: {new_run_id}')

train_cfg = Config(
    horizon           = HORIZON,
    xgb_n_trials      = TRIALS,
    skip_backtest     = False,
    save_outputs      = True,
    save_to_supabase  = True,
    paper_trade       = True,
    initial_capital   = CAPITAL,
    max_positions     = 10,
    model_version     = new_run_id,
    resolve_outcomes_on_start = False,   # already done in Cell 4
    rebalance_every   = HORIZON,
    embargo           = HORIZON,
)

new_stats, new_signals = run(train_cfg)
print(f'\nNew model — OOF IC={new_stats.get("oof_ic", "n/a")}  Sharpe={new_stats.get("Sharpe", "n/a")}  CAGR={new_stats.get("CAGR", "n/a")}')

In [ ]:
# ── Cell 6: Compare new model vs currently deployed ──────────────────────────
from src.models.improvement import compare_models

new_oof_ic = new_stats.get('oof_ic', float('nan'))
if new_oof_ic is None: new_oof_ic = float('nan')

should_deploy = compare_models(float(new_oof_ic), current_ic, min_improvement=0.005)

print(f'Current deployed IC : {"{:.4f}".format(current_ic) if not math.isnan(current_ic) else "n/a"}')
print(f'New model OOF IC    : {"{:.4f}".format(new_oof_ic) if not math.isnan(new_oof_ic) else "n/a"}')
print(f'Min improvement req : 0.0050')
print(f'Deploy decision     : {"✅ YES — deploying new model" if should_deploy else "❌ NO — keeping current model"}')

In [ ]:
# ── Cell 7: Deploy if better ─────────────────────────────────────────────────
import shutil
from pathlib import Path
from src.models.improvement import mark_deployed

if should_deploy:
    # Mark in Supabase + local JSON
    mark_deployed(new_run_id, sb, 'outputs', previous_run_id=current_run_id)

    # Save artifacts to this run's date-stamped folder: swing_outputs/<date>/
    run_path = Path(DRIVE_RUN)
    run_path.mkdir(parents=True, exist_ok=True)
    for folder in ('models', 'registry', 'outputs', 'data'):
        if Path(folder).exists():
            shutil.copytree(folder, str(run_path / folder), dirs_exist_ok=True)
    print(f'Model {new_run_id} deployed and saved to swing_outputs/{RUN_DATE}/')
    !ls -R "$DRIVE_RUN" | head -40
else:
    # Save new model to Drive as a candidate (for manual review)
    cand_path = Path(DRIVE_RUN) / 'candidates' / new_run_id
    cand_path.mkdir(parents=True, exist_ok=True)
    shutil.copy('models/best_params.json', str(cand_path / 'best_params.json'))
    if Path('registry').exists():
        shutil.copytree('registry', str(cand_path / 'registry'), dirs_exist_ok=True)
    print(f'Current model kept. New candidate saved to {cand_path}')

In [ ]:
# ── Cell 8: Generate next-week signals (fast mode, no re-training) ───────────
sig_cfg = Config(
    horizon           = HORIZON,
    xgb_n_trials      = 0,          # load saved best_params.json
    skip_backtest     = True,        # skip walk-forward (~18 min), only need signals
    save_outputs      = True,
    save_to_supabase  = True,
    paper_trade       = True,
    initial_capital   = CAPITAL,
    max_positions     = 10,
    model_version     = new_run_id if should_deploy else current_run_id,
    resolve_outcomes_on_start = False,
    rebalance_every   = HORIZON,
    embargo           = HORIZON,
)
_, week_signals = run(sig_cfg)
print(f'\nGenerated {len(week_signals)} signals for next week')
if not week_signals.empty:
    print(week_signals[['ticker','signal','prob_up','entry_price','stop_loss','target_price']].to_string(index=False))

In [ ]:
# ── Cell 9: Summary ─────────────────────────────────────────────────────────
from datetime import date
import math

long_count = int((week_signals.get('signal', '') == 'LONG').sum()) if not week_signals.empty else 0

print('═' * 60)
print('  WEEKLY RETRAIN SUMMARY')
print('═' * 60)
print(f'  Date                  : {date.today()}')
print(f'  New run_id            : {new_run_id}')
print(f'  Outcomes resolved     : {n_resolved}')
print(f'  Previous IC (4-wk)    : {"{:.4f}".format(current_ic) if not math.isnan(current_ic) else "n/a"}')
print(f'  New model OOF IC      : {"{:.4f}".format(float(new_oof_ic)) if not math.isnan(float(new_oof_ic)) else "n/a"}')
print(f'  Deployment decision   : {"DEPLOYED" if should_deploy else "KEPT PREVIOUS"}')
print(f'  Next-week signals     : {len(week_signals)} total  |  {long_count} LONG')
print('─' * 60)
print(f'  View dashboard        : https://your-app.streamlit.app')
print(f'  Next run              : Next Sunday morning')
print('═' * 60)

if not ic_series.empty:
    print('\nIC trend (last 8 weeks):')
    print(ic_series.to_string(index=False))